# StockMkt_R — Practice Skeleton

**Short name:** `StockMkt_R` (GitHub-friendly; full title *Modeling Stock Market Data* from Packt *Practical Data Science Cookbook*, 2nd ed., Ch. 4).

**Goal.** Acquire a Finviz-style snapshot, clean messy numerics, explore price outliers, build a 10-flag **relative valuation index**, screen a short target list, and overlay 50/200-day moving averages on historical prices.

**Not investment advice.** The book is explicit: this chapter teaches the *data-science pipeline* on market data. It will not make you a quant and it will not make you rich.

**Offline data.** Live Finviz `export.ashx` and the 2014 Yahoo `ichart.finance.yahoo.com` URLs from the book are brittle. Use `data/finviz.csv` and `data/historical_prices.csv` (bundled). The acquire cell still shows the original URL pattern.

**Companion files.** Solution notebook · `StockMkt_R_Cheatsheet.docx` · reusable template · 1-page report · memo · strategy guide · `stockmkt_r_flowchart.png`.


## 0. Setup

Load `ggplot2`, `plyr` (or `dplyr`), `reshape2` (or `tidyr`), `zoo`. Set a minimal theme.

Book packages:

```r
# install.packages(c("XML","ggplot2","plyr","reshape2","zoo"))
library(ggplot2); library(plyr); library(reshape2); library(zoo)
```


In [ ]:
# YOUR CODE HERE
# library(ggplot2)
# library(plyr)
# library(reshape2)
# library(zoo)
# theme_set(theme_minimal())

## 1. Acquire the snapshot

### Task 1–3

1. Read `data/finviz.csv` into `finviz`. Keep strings as strings (`stringsAsFactors = FALSE`).
2. `head(finviz[, 1:6])` and `dim(finviz)`.
3. *(Optional, live)* Reconstruct the book export URL with `sprintf` + `paste(0:68, collapse = ",")`. Do **not** depend on the live download for the rest of the notebook.

Book pattern:

```r
url_to_open <- sprintf("http://finviz.com/export.ashx?v=152&c=%s", paste(0:68, collapse = ","))
# finviz <- read.csv(url(url_to_open))
```


In [ ]:
# Task 1
finviz <- # YOUR CODE HERE

# Tasks 2–3
# YOUR CODE HERE

## 2. Summarize fields and learn the vocabulary

### Task 4–7

4. `summary(finviz[, 1:6])` — which `Sector` dominates?
5. List identifying fields: Ticker, Company, Sector, Industry, Country.
6. Write one sentence each for **Price, Volume, P/E, PEG, Debt/Equity, Beta, RSI**.
7. Why is *sector* coarser than *industry*? Give Apple (Consumer Goods / Electronic Equipment) as the book example — then pick one name from *this* file.


In [ ]:
# Tasks 4–7
# YOUR CODE HERE

## 3. Clean numerics (`clean_numeric`)

Imported snapshots hide `%`, `$`, commas, and parentheses inside "numeric" columns, so R treats them as character.

### Task 8–10

8. Write `clean_numeric <- function(s) { s <- gsub("%|\\$|,|\\)|\\(", "", s); as.numeric(s) }`.
9. Apply it to every column after the six identifiers. Book style:

```r
finviz <- cbind(finviz[, 1:6], apply(finviz[, 7:ncol(finviz)], 2, clean_numeric))
```

10. `str()` the cleaned frame. Confirm `Price`, `P.E` (or `P.E` after `make.names`), growth rates, and ownership are numeric.

**Column-index warning (book):** if the export gains columns, every hard-coded `7:68` breaks. Prefer names.


In [ ]:
# Tasks 8–10
clean_numeric <- function(s) {
  # YOUR CODE HERE
}

# finviz <- cbind(...)
# YOUR CODE HERE

## 4. Explore the price distribution

### Task 11–14

11. `hist(finviz$Price, breaks = 100)` — why is this chart useless?
12. Cap the axis: `hist(finviz$Price[finviz$Price < 150], breaks = 100)`.
13. Sector means with `aggregate(Price ~ Sector, data = finviz, FUN = mean)` and a `ggplot` bar.
14. Which sector looks expensive, and what do you *suspect* before drilling down?


In [ ]:
# Tasks 11–14
# YOUR CODE HERE

## 5. Drill into Financial → industry → company, then drop BRK-A

### Task 15–18

15. `aggregate(Price ~ Sector + Industry, ...)` then `subset(..., Sector == "Financial")`. Bar the industries.
16. Subset `Industry == "Property & Casualty Insurance"` and bar companies (rotate x labels 90°).
17. Name the outlier ticker and its price.
18. `finviz <- subset(finviz, Ticker != "BRK-A")` and recompute sector means. Did Financial come back to earth?


In [ ]:
# Tasks 15–18
# YOUR CODE HERE

## 6. Relative valuation — sector and industry averages

Relative valuation compares a name to *similar* names (sector / industry), not to an intrinsic DCF.

### Task 19–23

19. `sector_avg <- melt(finviz, id = "Sector")` then keep `Price`, `P.E`, `PEG`, `P.S`, `P.B` (use `names(finviz)` — `make.names` may have turned `P/E` into `P.E`).
20. `na.omit`, force `value` numeric, `dcast(sector_avg, Sector ~ variable, mean)`. Rename to `SAvgPE`, `SAvgPEG`, `SAvgPS`, `SAvgPB`, `SAvgPrice`.
21. Repeat at industry grain (`id = c("Sector","Industry")`) → `IAvg*`.
22. `merge` both average tables back onto `finviz`.
23. Why did `nrow` drop after the industry merge? Is that acceptable for a *screen*?


In [ ]:
# Tasks 19–23
# YOUR CODE HERE

## 7. Ten under-average flags → RelValIndex

### Task 24–27

24. Create ten 0-columns: `SPEUnder`, `SPEGUnder`, `SPSUnder`, `SPBUnder`, `SPriceUnder`, and the five `I*` twins.
25. Flip to 1 when the stock metric is **strictly below** the matching average.
26. `finviz$RelValIndex <- apply(finviz[flag_cols], 1, sum)` (or `rowSums`). Scale is 0–10.
27. `potentially_undervalued <- subset(finviz, RelValIndex >= 8)` and print Ticker / Company / index.


In [ ]:
# Tasks 24–27
# YOUR CODE HERE

## 8. Screen a target list and inspect histories

Book example filters (edit later in the simulation cell):

- Country == "USA"
- Price in (20, 100)
- Volume > 10,000
- EPS (ttm) > 0 and both growth fields > 0
- Total Debt/Equity < 1
- Beta < 1.5
- Institutional Ownership < 30
- RelValIndex > 8

### Task 28–32

28. `target_stocks <- subset(...)`. Goal: a *short* list (book aimed at < 10).
29. Read `data/historical_prices.csv` (offline stand-in for the Yahoo `ichart` loop).
30. For **one** symbol compute 50- and 200-day MAs with `zoo::rollmean(..., align = "right")` and line-plot AdjClose + both MAs.
31. Combined `qplot`/`ggplot` of AdjClose coloured by Symbol.
32. `ddply` (or `dplyr::summarise`) Open-first / High-max / Low-min / Close-last and a dodged or faceted bar chart.


In [ ]:
# Tasks 28–32
target_stocks <- # YOUR CODE HERE

hist_px <- # YOUR CODE HERE

# MA + charts
# YOUR CODE HERE

## Alternate code (same results)

Re-do **one** of these with a different stack:

- Sector means: `aggregate` ↔ `plyr::ddply` ↔ `dplyr::summarise`
- RelVal averages: `melt`/`dcast` ↔ `tidyr::pivot_longer`/`pivot_wider` ↔ `dplyr` group-mean + join
- Flags: ten assignments ↔ `across()` / a small helper
- Moving averages: `zoo::rollmean` ↔ `filter(rep(1/n, n), sides = 1)` ↔ `TTR::SMA` if installed


In [ ]:
# YOUR CODE HERE — pick one alternate

## More practice

1. Rebuild RelValIndex with **median** sector/industry stats instead of means. How many names still score ≥ 8?
2. Add a quality flag: `Beta < 1` *or* `RSI < 40` as extra points (index now 0–12).
3. Correlation heatmap of daily returns among the target symbols (wide `dcast` of AdjClose → `cor`). Which pair moves together?
4. Industry boxplots of `P.E` for Technology vs Utilities (drop `P.E > 200` first).


In [ ]:
# YOUR CODE HERE — at least two practice items

## Simulation / what-if

Knobs (edit and re-run):

- `idx_cut` — RelValIndex minimum (try 6, 8, 10)
- `price_lo`, `price_hi`
- `max_de` — Debt/Equity cap
- `max_beta`
- `max_inst`
- `usa_only` — TRUE/FALSE

Record: `n_pass`, median Price of passers, sector mix.

Optional noise: add `N(0, noise_sd)` to Price before rebuilding averages — does the index ranking stay stable?


In [ ]:
idx_cut <- 8
price_lo <- 20
price_hi <- 100
max_de <- 1
max_beta <- 1.5
max_inst <- 30
usa_only <- TRUE
noise_sd <- 0
set.seed(42)

# YOUR CODE HERE
# n_pass <- ...
# table(passers$Sector)

## Audience rewrite

Rewrite the *same* finding four ways (see attached audience PDFs):

Finding to adapt: *“A single Class-A share (BRK-A) pulled Financial-sector mean price into the stratosphere. After dropping it, sector means sit in a tight $25–45 band. A 10-flag relative-value index plus a hard screen left a short US list whose 50-day MAs can be compared with 200-day MAs.”*

1. **Expert / quant** — methods, bias from means vs medians, look-ahead, survivorship.
2. **Technician / screener operator** — exact filters, column names, how to re-run Monday.
3. **Executive / CIO** — headline, what changed when the outlier left, what the short list is *not*.
4. **Nonspecialist / retail reader** — no jargon; “cheap vs its neighbors, not a crystal ball.”


In [ ]:
# YOUR TEXT HERE

## Next

Open `StockMkt_R_Solution.ipynb` only after attempting each task. Keep `StockMkt_R_Cheatsheet.docx` beside the skeleton. Reuse `StockMkt_R_Reusable_Template.ipynb` for the next snapshot (ETFs, another market, next quarter).
